# 01 · Comprensión de los datos (OE1)

Fase 2 de CRISP-DM. Construye el diccionario de variables, describe el conjunto, aplica pruebas de asociación con la variable objetivo binaria y deja las tablas y figuras que van al documento.

In [ ]:
# Preparar el entorno (igual que en el cuaderno 00)
import os, sys
REPO_URL = "https://github.com/andresjaimes2004/tesis-desercion.git"
REPO_DIR = "/content/tesis-desercion"
if "google.colab" in sys.modules and not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import pandas as pd
from src import config, eda, plots
from src.data import load_raw, binarize_target
from src.utils import set_seed

set_seed()
pd.set_option("display.max_columns", 50)
config.REPORTS.mkdir(parents=True, exist_ok=True)

df = binarize_target(load_raw())
print(df.shape)

## 1. Diccionario de variables

In [ ]:
meta = eda.fetch_metadata()
diccionario = eda.build_dictionary(df, meta)
diccionario.to_csv(config.REPORTS / "diccionario_variables.csv", index=False, encoding="utf-8-sig")
print(diccionario["tipo"].value_counts())
diccionario

## 2. Variable objetivo

In [ ]:
print(df[config.BINARY_TARGET].value_counts(normalize=True).mul(100).round(1))
plots.target_distribution(df)

## 3. Estadísticos descriptivos

Numéricas: tendencia central y dispersión por grupo. Categóricas y binarias: tasa de deserción por categoría.

In [ ]:
tipos = {c: eda.variable_type(df, c) for c in eda.predictors(df)}
numericas = [c for c, t in tipos.items() if t == "numérica"]
desc = df.groupby(config.BINARY_TARGET)[numericas].agg(["mean", "median", "std"]).T.round(2)
desc.to_csv(config.REPORTS / "descriptivos_numericas.csv", encoding="utf-8-sig")
desc

## 4. Pruebas de asociación con la deserción

- Categóricas y binarias: chi cuadrado de independencia con V de Cramér. Las categorías con menos de 30 registros se agrupan en "Otras" y se reporta el porcentaje de frecuencias esperadas menores a 5 (Cochran, 1954).
- Numéricas: U de Mann-Whitney con correlación rango-biserial como tamaño del efecto (Kerby, 2014).
- Los p-valores se ajustan por comparaciones múltiples con Benjamini-Hochberg (1995).

In [ ]:
asoc = eda.association_table(df)
asoc.to_csv(config.REPORTS / "asociacion_variables.csv", index=False, encoding="utf-8-sig")
print("Significativas tras el ajuste:", int(asoc["significativa"].sum()), "de", len(asoc))
asoc[["variable", "tipo", "momento", "prueba", "p_ajustado_bh", "efecto", "direccion", "pct_esperadas_menor_5"]].round(4)

In [ ]:
plots.effect_sizes(asoc)

In [ ]:
# Las cuatro numéricas y las tres categóricas con mayor efecto
top_num = asoc.query("tipo == 'numérica'")["variable"].head(4).tolist()
top_cat = asoc.query("tipo != 'numérica'")["variable"].head(3).tolist()
plots.numeric_by_group(df, top_num)
tasas = {c: eda.dropout_rate_by_category(df, c) for c in top_cat}
for c, t in tasas.items():
    print(c); display(t.round(3))
plots.categorical_rates(tasas)

## 5. Redundancia entre variables numéricas

In [ ]:
redundantes = eda.redundant_pairs(df)
redundantes.to_csv(config.REPORTS / "pares_redundantes.csv", index=False, encoding="utf-8-sig")
plots.spearman_heatmap(df, numericas)
redundantes.round(3)

## 6. Resumen para el documento

Ejecutar esta celda y enviar su salida completa.

In [ ]:
print("VARIABLES:", len(asoc), "| significativas (BH < 0,05):", int(asoc['significativa'].sum()))
print("\nNO SIGNIFICATIVAS:")
print(asoc.loc[~asoc['significativa'], ['variable', 'p_ajustado_bh', 'efecto']].round(4).to_string(index=False))
print("\nTOP 15 POR TAMAÑO DEL EFECTO:")
cols = ['variable', 'tipo', 'efecto', 'direccion', 'mediana_deserta', 'mediana_no_deserta', 'pct_esperadas_menor_5']
print(asoc[cols].head(15).round(3).to_string(index=False))
print("\nPARES REDUNDANTES:")
print(redundantes.round(3).to_string(index=False))
for c, t in tasas.items():
    print(f"\nTASA DE DESERCIÓN POR CATEGORÍA – {c}:")
    print(t.round(3).to_string())

## 7. Guardar resultados en GitHub

Descargar la carpeta `reports/` (CSV y figuras) desde el panel de archivos y subirla al repositorio con el mensaje `Fase 2: comprensión de los datos`. Guardar también este cuaderno con Archivo > Guardar una copia en GitHub.